In [3]:
# Install dependencies with FlashAttention 2 for faster inference
!pip install ninja packaging wheel
!pip install -U flash-attn xformers

# Install TTS and audio packages
!pip install -q qwen-tts soundfile gradio
!apt-get install -qq sox libsox-fmt-all

print("✅ All dependencies installed successfully!")
print("   - FlashAttention 2 (faster generation)")
print("   - Qwen-TTS package")
print("   - SoundFile (audio processing)")
print("   - Gradio (web interface)")

  Using cached flash_attn-2.8.3.post1.tar.gz (8.5 MB)
  Preparing metadata (setup.py) ... done
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl (3.3 MB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
  Running setup.py clean for flash-attn
Failed to build flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)
✅ All dependencies installed successfully!
   - FlashAttention 2 (faster generation)
   - Qwen-TTS package
   - SoundFile (audio processing)
   - Gradio (web interface)


In [4]:
import torch
import soundfile as sf
import gradio as gr
import numpy as np
from qwen_tts import Qwen3TTSModel

print("Loading Qwen3-TTS Base model...")
print("(This takes 3-5 minutes on first run)")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",  # FlashAttention 2 is disabled due to installation issues
)

print("✅ Model loaded successfully!")
print("   You can now clone voices in the interface below.")

Loading Qwen3-TTS Base model...
(This takes 3-5 minutes on first run)


generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

configuration.json:   0%|          | 0.00/76.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Model loaded successfully!
   You can now clone voices in the interface below.


In [8]:
def clone_voice(new_text, language, reference_audio, ref_transcript):
    """
    Clone a voice and generate speech in that cloned voice.

    Args:
        new_text: The text you want the cloned voice to speak
        language: Target language for synthesis
        reference_audio: Audio file path (3-10 seconds of clear speech)
        ref_transcript: Exact transcript of what's said in reference audio
    """
    try:
        if reference_audio is None:
            return None, "❌ Por favor, sube un archivo de audio de referencia (se recomiendan 3-10 segundos)"

        if not ref_transcript or ref_transcript.strip() == "":
            return None, "❌ Por favor, proporciona la transcripción de tu audio de referencia"

        # Generate voice clone using official Qwen3-TTS API
        wavs, sr = model.generate_voice_clone(
            text=new_text,
            language=language,
            ref_audio=reference_audio,
            ref_text=ref_transcript,
        )

        # Handle output format
        if isinstance(wavs, (list, tuple)):
            audio_data = np.array(wavs[0])
        else:
            audio_data = np.array(wavs)

        return (int(sr), audio_data), f"✅ Voz clonada exitosamente! Frecuencia de muestreo: {sr}Hz"

    except Exception as e:
        import traceback
        return None, f"❌ Error: {str(e)}\n\n{traceback.format_exc()}"

# Supported languages (10 major languages)
languages = [
    "Chinese", "English", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

# Create Gradio interface
interface = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Textbox(
            label="📝 NUEVO Texto (lo que quieres que diga la voz clonada)",
            placeholder="Introduce el texto que quieres que la voz clonada pronuncie...",
            lines=4,
            value="Hola! Esta es mi voz clonada diciendo nuevas palabras.",
            interactive=True # Explicitly set to interactive
        ),
        gr.Dropdown(
            choices=languages,
            value="Spanish", # Set default to Spanish
            label="🌐 Idioma",
            interactive=True
        ),
        gr.Audio(
            label="🎤 Audio de Referencia (3-10 segundos de habla clara)",
            type="filepath",
            sources=["upload", "microphone"],
            interactive=True
        ),
        gr.Textbox(
            label="📄 Transcripción del Audio de Referencia",
            placeholder="Escribe EXACTAMENTE lo que se dice en el audio de referencia...",
            lines=3,
            value="",
            interactive=True # Explicitly set to interactive
        )
    ],
    outputs=[
        gr.Audio(label="🔊 Salida de Voz Clonada", type="numpy"),
        gr.Textbox(label="📊 Estado", lines=2)
    ],
    title="🎙️ Clonación de Voz Qwen3-TTS",
    description="""
    **¡Clona cualquier voz a partir de solo 3 segundos de audio!**

    **Cómo usar:**
    1. Sube un clip de audio de 3-10 segundos de la voz que quieres clonar
    2. Escribe exactamente lo que se dice en ese audio (la transcripción)
    3. Introduce el nuevo texto que quieres que esta voz pronuncie
    4. Haz clic en 'Enviar' y espera tu voz clonada!

    **Consejos para mejores resultados:**
    - Usa audio claro sin ruido de fondo
    - Asegúrate de que tu transcripción coincida exactamente con el audio
    - Una referencia de audio más larga (10-20 segundos) da mejores resultados
    - ¡Funciona en 10 idiomas!
    """,
    examples=[
        [
            "Bienvenidos a mi demo de clonación de voz con IA!",
            "Spanish",
            None,
            ""
        ],
        [
            "Esta tecnología es increíble!",
            "Spanish",
            None,
            ""
        ]
    ]
)

# Launch the interface
print("🚀 Lanzando interfaz de Gradio...")
interface.launch(share=True, debug=True) # Changed debug to True

🚀 Lanzando interfaz de Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d629e1c3038c066ba0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e49d093bef986112b5.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://4627a108c43ca92819.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://a75b344a6a7b84a0ce.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://d629e1c3038c066ba0.gradio.live
